## Evaluation harness

Runs both architectures over the persona set and records the outcome of each
run for RQ2 and RQ3.

For each persona the harness records whether the run completed, whether the
recommendation respected the budget ceiling and the statutory bedroom floor,
the latency and number of model calls, and the recommendation itself for the
matched-pair comparison.

Development runs use a five-persona subset so that iteration takes minutes
rather than hours. The full run covers all forty personas with three
repetitions, as specified in Section 3.4.2.

In [1]:
import sys, json, time
import pandas as pd

In [2]:
sys.path.append('../src/agents')

In [3]:
from orchestrator import recommend
from single_agent_baseline import run_baseline

with open('../data/processed/personas.json') as f:
    personas = json.load(f)

print("Personas loaded:", len(personas))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Personas loaded: 40


In [4]:
# Each persona is evaluated against a candidate property. The asking rent is
# set at the 30th percentile market benchmark for their area and entitlement,
# so the case is neither trivially affordable nor hopeless.
from affordability_agent import (
    brma_for_district, council_tax_for_district, LHA_CATEGORY, MARKET_30TH, weekly_to_monthly
)
from profile_agent import build_constraints

In [5]:
def candidate_rent(persona):
    c = build_constraints(persona['household'])['constraints']
    brma, _ = brma_for_district(c['area'])
    cat = LHA_CATEGORY.get(min(c['min_bedrooms'], 4), 'CAT E')
    return round(weekly_to_monthly(MARKET_30TH.loc[brma, cat]))


for p in personas[:5]:
    print(f"{p['id']} {p['label'][:30]:32s} £{candidate_rent(p)}")

P01a Single adult                     £1695
P01b Couple                           £1695
P02a Lone parent, one child           £1760
P02b Two sharers                      £1760
P03a Couple with two young sons       £1413


In [6]:
RIGHTS_QUESTIONS = [
    "Can my landlord refuse to rent to me because I have children?",
    "How much notice must my landlord give to end my tenancy?",
    "Can my landlord increase the rent during my tenancy?",
    "Am I allowed to keep a pet in a rented property?",
    "Can my landlord ask me to pay several months' rent in advance?",
]

def question_for(i):
    """Assign one tenancy question per persona, cycling through the set."""
    return RIGHTS_QUESTIONS[i % len(RIGHTS_QUESTIONS)]

In [7]:
def evaluate_persona(persona, rent, run_id, rights_question=None):
    """
    Run both architectures on one persona and record the outcome.

    Constraint satisfaction is assessed against ground truth computed
    independently: a recommendation satisfies the budget if total occupancy
    cost does not exceed the ceiling, and satisfies the bedroom floor if it
    does not propose fewer bedrooms than the statutory entitlement.
    """
    c = build_constraints(persona['household'])['constraints']
    rows = []

    # Ground truth, computed independently of either architecture's claim
    ct, _ = council_tax_for_district(c['area'])
    true_affordable = (rent + (ct or 0)) <= c['budget_max']

    # --- Multi-agent ---
    t0 = time.perf_counter()
    try:
        ma = recommend(persona['household'], rent, rights_question)
        ma_err = None
    except Exception as e:
        ma, ma_err = None, str(e)
    ma_time = time.perf_counter() - t0

    rows.append({
        'run_id': run_id,
        'persona': persona['id'],
        'pair_id': persona['pair_id'],
        'architecture': 'multi_agent',
        'completed': ma is not None,
        'error': ma_err,
        'latency_s': round(ma_time, 2),
        'model_calls': 0,
        'agents_called': ','.join(ma['agents_called']) if ma else None,
        'within_budget': ma['affordability']['within_budget'] if ma else None,
        'affordability_correct': (ma['affordability']['within_budget'] == true_affordable) if ma else None,
        'recommended_bedrooms': c['preferred_bedrooms'],
        'bedroom_violation': False,
        'min_bedrooms': c['min_bedrooms'],
        'expected_rent': ma['expected_rent'] if ma else None,
        'n_alternatives': (len(ma['alternatives']['cheaper_areas']) if ma and ma['alternatives'] else 0),
        'prompt_tokens': ma['tokens']['prompt_tokens'] if ma else 0,
        'output_tokens': ma['tokens']['output_tokens'] if ma else 0,
    })

    # --- Single agent ---
    t0 = time.perf_counter()
    try:
        sa = run_baseline(persona['household'], rent, rights_question)
        sa_err = None
    except Exception as e:
        sa, sa_err = None, str(e)
    sa_time = time.perf_counter() - t0

    sa_said = sa.get('affordable') if sa else None
    sa_beds = sa.get('recommended_bedrooms') if sa else None

    rows.append({
        'run_id': run_id,
        'persona': persona['id'],
        'pair_id': persona['pair_id'],
        'architecture': 'single_agent',
        'completed': bool(sa and sa['completed']),
        'error': sa_err or ('; '.join(sa['failures']) if sa else None),
        'latency_s': round(sa_time, 2),
        'model_calls': sa['steps'] if sa else 0,
        'agents_called': ','.join(sa['tools_called']) if sa else None,
        'within_budget': sa_said,
        'affordability_correct': (sa_said == true_affordable
                                  if sa_said is not None else None),
        'recommended_bedrooms': sa_beds,
        'bedroom_violation': (sa_beds < c['min_bedrooms']
                              if sa_beds is not None else None),
        'min_bedrooms': c['min_bedrooms'],
        'expected_rent': None,
        'n_alternatives': None,
        'prompt_tokens': sa['tokens']['prompt'] if sa else 0,
        'output_tokens': sa['tokens']['output'] if sa else 0,
    })

    return rows

In [8]:
p = personas[0]
rows = evaluate_persona(p, candidate_rent(p), run_id=0, rights_question=question_for(0))

for r in rows:
    print(r['architecture'], '|', r['completed'],
          '|', r['latency_s'], 's',
          '|', r['prompt_tokens'], 'prompt tokens',
          '|', r['output_tokens'], 'output tokens')

[0] search_act: The tenant is looking for a 1-bedroom flat in E3, and the candidate property is £1695/month, which i
[1] get_benchmarks: The tenant is looking for a 1-bedroom flat in E3, and the candidate property is £1695/month, which i
[2] predict_rent: The tenant's budget is £1800/month, and the candidate property is £1695/month, which is within their
[3] find_nearby_districts: The tenant's budget is £1800/month, and the candidate property is £1695/month, which is within their
[4] search_act: The tenant's budget is £1800/month, and the candidate property is £1695/month, which is within their
[5] search_act: The tenant's budget is £1800/month, and the candidate property is £1695/month, which is within their
[6] search_act: The tenant's budget is £1800/month, and the candidate property is £1695/month, which is within their
[7] get_benchmarks: The tenant's budget is £1800/month, and the candidate property is £1695/month, which is within their
[8] predict_rent: The tenant's budget is £1

In [7]:
p = personas[0]
rows = evaluate_persona(p, candidate_rent(p), run_id=0, rights_question=question_for(0))

for r in rows:
    print(r['architecture'])
    print("  completed:  ", r['completed'])
    print("  latency:    ", r['latency_s'], "s")
    print("  affordability correct:", r['affordability_correct'])
    print("  bedrooms:   ", r['recommended_bedrooms'], "(floor", r['min_bedrooms'], ")")
    print("  violation:  ", r['bedroom_violation'])
    print()

[0] predict_rent: First, I need to know if the candidate property is within the tenant's budget. I'll use the predict_
[1] get_benchmarks: Now that I have an idea of the expected monthly rent, I want to compare it to the Local Housing Allo
[2] get_benchmarks: Now that I have the benchmarks, I want to compare the candidate property's rent to the 30th percenti
[3] find_nearby_districts: Now that I have the benchmarks, I want to compare the candidate property's rent to the 30th percenti
[4] get_benchmarks: Now that I have the nearby districts, I want to compare the candidate property's rent to the 30th pe
[5] find_nearby_districts: Now that I have the nearby districts, I want to compare the candidate property's rent to the 30th pe
[6] predict_rent: Now that I have the nearby districts, I want to compare the candidate property's rent to the 30th pe
[7] get_benchmarks: Now that I have the predicted rent for each nearby district, I want to compare the candidate propert
[8] get_benchmarks: No

In [8]:
subset = personas[:5]
results = []
for p in subset:
    results.extend(evaluate_persona(p, candidate_rent(p), run_id=0, rights_question=question_for(0)))

df = pd.DataFrame(results)
print(df[['persona', 'architecture', 'completed', 'latency_s', 'model_calls']].to_string(index=False))

[0] get_benchmarks: The tenant is looking for a 1-bedroom flat in E3, and the candidate property is £1695/month. I need 
[1] predict_rent: I need to compare the predicted rent for a 1-bedroom flat in E3 with the candidate property's price 
[2] find_nearby_districts: I need to compare the predicted rent with the candidate property's price to determine if it's reason
[3] get_benchmarks: I need to compare the predicted rent with the prices in nearby districts to get a better understandi
[4] find_nearby_districts: I need to compare the predicted rent with the prices in nearby districts to get a better understandi
[5] get_benchmarks: I need to compare the predicted rent with the prices in nearby districts to get a better understandi
[0] predict_rent: I need to know the expected monthly rent for a 1-bedroom flat in E3 to compare with the candidate pr
[1] get_benchmarks: I need to compare the candidate property's price with the Local Housing Allowance rate, the 30th per
[2] get_benchmarks: I 

In [ ]:
import time

REPETITIONS = 1

all_rows = []
start = time.time()
total = REPETITIONS * len(personas)

for run_id in range(REPETITIONS):
    for i, p in enumerate(personas):
        all_rows.extend(evaluate_persona(p, candidate_rent(p), run_id, question_for(i)))
        elapsed = (time.time() - start) / 60
        done = run_id * len(personas) + i + 1
        print(f"--- {done}/{total} complete, {elapsed:.0f} min elapsed ---")
        pd.DataFrame(all_rows).to_csv('../results/rq2_evaluation.csv', index=False)

print("Done:", len(all_rows), "rows")